# 📊 Trader Performance vs Market Sentiment
## Primetrade.ai — Data Science Intern Assignment
**Objective:** Analyze how Bitcoin Fear/Greed Index relates to trader behavior and performance on Hyperliquid.

| | |
|---|---|
| **Datasets** | Fear/Greed Index (2644 rows) · Hyperliquid Trades (211,224 rows) |
| **Overlap** | 479 days (May 2023 – May 2025) |
| **Accounts** | 32 unique traders |
| **Coins** | 246 unique assets |

---
**Notebook Structure**
1. Setup & Imports
2. Data Loading & Overview
3. Cleaning & Feature Engineering
4. Part A — Exploratory Analysis
5. Part B — Sentiment vs Performance Analysis
6. Part C — Trader Segmentation
7. Insights Summary
8. Strategy Recommendations
9. Bonus — Predictive Model & Clustering


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────────────────
FEAR_C   = '#ef4444'
GREED_C  = '#22c55e'
NEUTRAL_C = '#a78bfa'
PALETTE  = {'Fear': FEAR_C, 'Greed': GREED_C, 'Neutral': NEUTRAL_C}

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e', 'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc', 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'xtick.color': '#aaa', 'ytick.color': '#aaa', 'text.color': '#eee',
    'grid.color': '#2a2a3e', 'grid.linestyle': '--', 'grid.alpha': 0.6,
    'legend.facecolor': '#1a1a2e', 'legend.edgecolor': '#444',
})

print("✅ All libraries loaded")
print(f"  pandas {pd.__version__}  |  numpy {np.__version__}")


## 2. Data Loading & Overview

In [ ]:
# ── Load raw files ───────────────────────────────────────────────────────────
fg_raw = pd.read_csv("fear_greed_index.csv")
td_raw = pd.read_csv("historical_data.csv")

print("=" * 60)
print("FEAR / GREED INDEX")
print("=" * 60)
print(fg_raw.head(3).to_string())
print(f"\nShape      : {fg_raw.shape}")
print(f"Columns    : {list(fg_raw.columns)}")
print(f"Missing    : {fg_raw.isnull().sum().to_dict()}")
print(f"Duplicates : {fg_raw.duplicated().sum()}")
print(f"\nClassification counts:\n{fg_raw['classification'].value_counts()}")


In [ ]:
print("=" * 60)
print("HYPERLIQUID TRADER DATA")
print("=" * 60)
print(td_raw.head(3).to_string())
print(f"\nShape      : {td_raw.shape}")
print(f"Columns    : {list(td_raw.columns)}")
print(f"\nMissing values:\n{td_raw.isnull().sum()}")
print(f"Duplicates : {td_raw.duplicated().sum()}")
print(f"\nUnique accounts : {td_raw['Account'].nunique()}")
print(f"Unique coins    : {td_raw['Coin'].nunique()}")
print(f"\nClosed PnL stats:\n{td_raw['Closed PnL'].describe()}")
print(f"\nDirection unique values: {td_raw['Direction'].unique().tolist()}")


## 3. Data Cleaning & Feature Engineering

In [ ]:
# ════════════════════════════════════════════
#  CLEAN FEAR / GREED
# ════════════════════════════════════════════
fg = fg_raw.copy()
fg['date'] = pd.to_datetime(fg['date'])
fg = fg.drop_duplicates(subset='date').sort_values('date').reset_index(drop=True)

# Simplify to 3-class sentiment
def simplify_sentiment(val):
    v = str(val).lower()
    if 'greed' in v:  return 'Greed'
    if 'fear'  in v:  return 'Fear'
    return 'Neutral'

fg['sentiment'] = fg['classification'].apply(simplify_sentiment)
fg['is_greed']  = (fg['sentiment'] == 'Greed').astype(int)

print(f"✅ Fear/Greed cleaned: {len(fg)} rows")
print(f"   Date range: {fg['date'].min().date()} → {fg['date'].max().date()}")
print(f"   Sentiment split (3-class):\n{fg['sentiment'].value_counts()}")


In [ ]:
# ════════════════════════════════════════════
#  CLEAN TRADER DATA
# ════════════════════════════════════════════
td = td_raw.copy()

# Parse IST timestamp  (format: DD-MM-YYYY HH:MM)
td['datetime'] = pd.to_datetime(td['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
td = td.dropna(subset=['datetime'])
td['date'] = td['datetime'].dt.normalize()   # date only

# Standardise direction to Long / Short / Other
def clean_direction(d):
    d = str(d).lower()
    if 'long'  in d or d == 'buy':  return 'Long'
    if 'short' in d or d == 'sell': return 'Short'
    return 'Other'

td['dir_clean'] = td['Direction'].apply(clean_direction)

# Keep only actionable trade events
EXCLUDE = ['Spot Dust Conversion', 'Auto-Deleveraging', 'Settlement']
td_trades = td[~td['Direction'].isin(EXCLUDE)].copy()

# Win flag
td_trades['is_win'] = (td_trades['Closed PnL'] > 0).astype(int)

print(f"✅ Trader data cleaned: {len(td_trades):,} trade rows (removed {len(td)-len(td_trades):,} non-trades)")
print(f"   Date range: {td_trades['date'].min().date()} → {td_trades['date'].max().date()}")
print(f"   Unique accounts: {td_trades['Account'].nunique()}")
print(f"\n   Direction breakdown:\n{td_trades['dir_clean'].value_counts()}")


In [ ]:
# ════════════════════════════════════════════
#  MERGE ON DATE
# ════════════════════════════════════════════
merged = td_trades.merge(
    fg[['date', 'sentiment', 'is_greed', 'value']].rename(columns={'value': 'fg_value'}),
    on='date', how='inner'
)

print(f"✅ Merged dataset: {len(merged):,} rows | {merged['date'].nunique()} days")
print(f"   Sentiment distribution in merged data:")
print(merged['sentiment'].value_counts())


In [ ]:
# ════════════════════════════════════════════
#  FEATURE ENGINEERING
# ════════════════════════════════════════════

# ── Daily market-level aggregates ─────────────────────
daily_mkt = merged.groupby(['date', 'sentiment', 'fg_value']).agg(
    total_pnl     = ('Closed PnL',  'sum'),
    avg_pnl       = ('Closed PnL',  'mean'),
    n_trades      = ('Closed PnL',  'count'),
    win_rate      = ('is_win',       'mean'),
    avg_size_usd  = ('Size USD',     'mean'),
    total_vol_usd = ('Size USD',     'sum'),
    long_ratio    = ('dir_clean',    lambda x: (x=='Long').mean()),
).reset_index()

# ── Per-account daily aggregates ──────────────────────
acct_daily = merged.groupby(['Account', 'date', 'sentiment']).agg(
    daily_pnl    = ('Closed PnL', 'sum'),
    n_trades     = ('Closed PnL', 'count'),
    win_rate     = ('is_win',      'mean'),
    avg_size_usd = ('Size USD',    'mean'),
    long_ratio   = ('dir_clean',   lambda x: (x=='Long').mean()),
).reset_index()

# ── Per-account overall metrics ────────────────────────
acct_stats = merged.groupby('Account').agg(
    total_pnl    = ('Closed PnL', 'sum'),
    avg_pnl      = ('Closed PnL', 'mean'),
    total_trades = ('Closed PnL', 'count'),
    win_rate     = ('is_win',      'mean'),
    avg_size_usd = ('Size USD',    'mean'),
    pnl_std      = ('Closed PnL', 'std'),
    long_ratio   = ('dir_clean',   lambda x: (x=='Long').mean()),
).reset_index()
acct_stats['sharpe_proxy'] = acct_stats['avg_pnl'] / (acct_stats['pnl_std'] + 1e-9)
acct_stats['consistency']  = acct_stats['win_rate']

# Drawdown proxy: max cumulative loss streak per account
def drawdown_proxy(pnl_series):
    cum = pnl_series.cumsum()
    return (cum - cum.cummax()).min()

acct_drawdown = merged.groupby('Account')['Closed PnL'].apply(drawdown_proxy).reset_index()
acct_drawdown.columns = ['Account', 'max_drawdown']
acct_stats = acct_stats.merge(acct_drawdown, on='Account')

print(f"✅ Feature engineering complete")
print(f"   daily_mkt  : {daily_mkt.shape}")
print(f"   acct_daily : {acct_daily.shape}")
print(f"   acct_stats : {acct_stats.shape}")
print(f"\nAccount-level sample:")
print(acct_stats[['Account','total_pnl','win_rate','total_trades','sharpe_proxy']].head())


## 4. Part A — Exploratory Data Analysis

In [ ]:
# ── Fig 1: Fear/Greed distribution over time ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fear/Greed Index Overview", fontsize=15, fontweight='bold', color='white', y=1.02)

# Pie chart
counts = fg['sentiment'].value_counts()
colors = [PALETTE[s] for s in counts.index]
axes[0].pie(counts, labels=counts.index, colors=colors, autopct='%1.1f%%',
            startangle=140, textprops={'color':'white','fontsize':11},
            wedgeprops={'linewidth':1.5, 'edgecolor':'#0f0f1a'})
axes[0].set_title("Sentiment Distribution (2018–2025)")

# Time series rolling average
fg_plot = fg.set_index('date')
axes[1].fill_between(fg_plot.index, fg_plot['value'],
                     where=fg_plot['value'] >= 50, color=GREED_C, alpha=0.4, label='Greed zone')
axes[1].fill_between(fg_plot.index, fg_plot['value'],
                     where=fg_plot['value'] < 50,  color=FEAR_C,  alpha=0.4, label='Fear zone')
axes[1].plot(fg_plot.index, fg_plot['value'].rolling(30).mean(), color='white', lw=1.5, label='30d MA')
axes[1].axhline(50, color='#aaa', lw=0.8, linestyle='--')
axes[1].set_title("Fear/Greed Index Over Time")
axes[1].set_ylabel("Index Value")
axes[1].legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig("fig1_fg_overview.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 1 saved → fig1_fg_overview.png")


In [ ]:
# ── Fig 2: Trader data overview ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Trader Data Overview", fontsize=15, fontweight='bold', color='white')

# PnL distribution (trimmed)
pnl_trim = merged['Closed PnL'].clip(-2000, 2000)
axes[0].hist(pnl_trim, bins=80, color=NEUTRAL_C, alpha=0.8, edgecolor='#0f0f1a')
axes[0].axvline(0, color='white', lw=1.5, linestyle='--')
axes[0].set_title("Closed PnL Distribution (clipped ±2000)")
axes[0].set_xlabel("PnL (USD)")
axes[0].set_ylabel("Count")

# Trade count by coin (top 10)
top_coins = merged['Coin'].value_counts().head(10)
axes[1].barh(top_coins.index[::-1], top_coins.values[::-1], color=NEUTRAL_C, alpha=0.8)
axes[1].set_title("Top 10 Most Traded Coins")
axes[1].set_xlabel("Number of Trades")

# Direction split
dir_counts = merged['dir_clean'].value_counts()
colors_dir = [GREED_C if d=='Long' else FEAR_C if d=='Short' else '#888' for d in dir_counts.index]
axes[2].bar(dir_counts.index, dir_counts.values, color=colors_dir, alpha=0.85, edgecolor='#0f0f1a')
axes[2].set_title("Trade Direction Distribution")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.savefig("fig2_trader_overview.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 2 saved → fig2_trader_overview.png")


## 5. Part B — Sentiment vs Performance Analysis
### Q1: Does performance differ between Fear vs Greed days?


In [ ]:
# ── Statistical comparison: Fear vs Greed PnL ────────────────────────────────
fear_pnl  = daily_mkt[daily_mkt['sentiment']=='Fear']['total_pnl']
greed_pnl = daily_mkt[daily_mkt['sentiment']=='Greed']['total_pnl']
t_stat, p_val = stats.ttest_ind(fear_pnl, greed_pnl)

print("╔══════════════════════════════════════════════════╗")
print("║  Daily PnL — Fear vs Greed (Statistical Test)   ║")
print("╠══════════════════════════════════════════════════╣")
for sent in ['Fear','Neutral','Greed']:
    grp = daily_mkt[daily_mkt['sentiment']==sent]['total_pnl']
    print(f"║  {sent:<8} | Mean: ${grp.mean():>8.1f}  | Median: ${grp.median():>7.1f}  ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  t-statistic : {t_stat:.4f}                              ║")
print(f"║  p-value     : {p_val:.4f}  {'(SIGNIFICANT ✓)' if p_val<0.05 else '(not significant)'}         ║")
print("╚══════════════════════════════════════════════════╝")


In [ ]:
# ── Fig 3: PnL & Win Rate by Sentiment ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Performance Metrics by Market Sentiment", fontsize=14, fontweight='bold', color='white')

order = ['Fear', 'Neutral', 'Greed']
colors = [PALETTE[s] for s in order]

# Avg daily PnL
means = [daily_mkt[daily_mkt['sentiment']==s]['total_pnl'].mean() for s in order]
bars  = axes[0].bar(order, means, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[0].axhline(0, color='white', lw=0.8, linestyle='--')
axes[0].set_title("Avg Daily Total PnL")
axes[0].set_ylabel("USD")
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'${val:,.0f}', ha='center', color='white', fontsize=9)

# Win rate boxplot
data_wr = [daily_mkt[daily_mkt['sentiment']==s]['win_rate'].dropna() for s in order]
bp = axes[1].boxplot(data_wr, labels=order, patch_artist=True,
                     medianprops={'color':'white','linewidth':2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for element in ['whiskers','caps','fliers']:
    for item in bp[element]: item.set_color('#aaa')
axes[1].set_title("Daily Win Rate Distribution")
axes[1].set_ylabel("Win Rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

# Avg trade size
sizes = [daily_mkt[daily_mkt['sentiment']==s]['avg_size_usd'].mean() for s in order]
axes[2].bar(order, sizes, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[2].set_title("Avg Trade Size (USD)")
axes[2].set_ylabel("USD")

plt.tight_layout()
plt.savefig("fig3_performance_by_sentiment.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 3 saved → fig3_performance_by_sentiment.png")


### Q2: Do traders change behavior based on sentiment?

In [ ]:
# ── Fig 4: Behavioral shifts by sentiment ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Trader Behavior Shifts by Market Sentiment", fontsize=14, fontweight='bold', color='white')

order = ['Fear', 'Neutral', 'Greed']
colors = [PALETTE[s] for s in order]

# Trade frequency
freq = [daily_mkt[daily_mkt['sentiment']==s]['n_trades'].mean() for s in order]
axes[0,0].bar(order, freq, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[0,0].set_title("Avg Trade Frequency per Day")
axes[0,0].set_ylabel("# Trades")
for i,(o,f) in enumerate(zip(order,freq)):
    axes[0,0].text(i, f+2, f'{f:.0f}', ha='center', color='white', fontsize=9)

# Long/Short ratio
lr = [daily_mkt[daily_mkt['sentiment']==s]['long_ratio'].mean() for s in order]
axes[0,1].bar(order, lr, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[0,1].axhline(0.5, color='white', lw=1, linestyle='--', label='50/50 line')
axes[0,1].set_title("Long Ratio (higher = more bullish)")
axes[0,1].set_ylabel("Proportion of Longs")
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
axes[0,1].legend(fontsize=8)

# Volume
vol = [daily_mkt[daily_mkt['sentiment']==s]['total_vol_usd'].mean() for s in order]
axes[1,0].bar(order, vol, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[1,0].set_title("Avg Daily Volume (USD)")
axes[1,0].set_ylabel("USD")
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# PnL volatility (std)
pnl_vol = [daily_mkt[daily_mkt['sentiment']==s]['total_pnl'].std() for s in order]
axes[1,1].bar(order, pnl_vol, color=colors, alpha=0.85, edgecolor='#0f0f1a', width=0.5)
axes[1,1].set_title("PnL Volatility (Std Dev) by Sentiment")
axes[1,1].set_ylabel("USD Std Dev")

plt.tight_layout()
plt.savefig("fig4_behavior_by_sentiment.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 4 saved → fig4_behavior_by_sentiment.png")


### Q3: Drawdown proxy by sentiment

In [ ]:
# ── Drawdown analysis ────────────────────────────────────────────────────────
fear_daily  = acct_daily[acct_daily['sentiment']=='Fear']['daily_pnl']
greed_daily = acct_daily[acct_daily['sentiment']=='Greed']['daily_pnl']

print("╔════════════════════════════════════════════════════════╗")
print("║           Drawdown Proxy — Fear vs Greed Days          ║")
print("╠════════════════════════════════════════════════════════╣")
print(f"║  Fear  days — Mean daily PnL/account : ${fear_daily.mean():>8.2f}       ║")
print(f"║  Greed days — Mean daily PnL/account : ${greed_daily.mean():>8.2f}       ║")
print(f"║  Fear  days — % days with loss       : {(fear_daily<0).mean()*100:>6.1f}%         ║")
print(f"║  Greed days — % days with loss       : {(greed_daily<0).mean()*100:>6.1f}%         ║")
print("╚════════════════════════════════════════════════════════╝")

# Overall drawdown per account
print("\nAccount Max Drawdown Summary:")
print(acct_stats[['Account','max_drawdown','total_pnl','win_rate']].sort_values('max_drawdown').head(10).to_string())


## 6. Part C — Trader Segmentation

In [ ]:
# ════════════════════════════════════════════
# SEGMENT 1: Winners vs Losers
# ════════════════════════════════════════════
acct_stats['segment_pnl'] = pd.cut(
    acct_stats['total_pnl'],
    bins=[-np.inf, -1000, 0, 5000, np.inf],
    labels=['Big Loser','Small Loser','Small Winner','Big Winner']
)

print("Segment 1 — PnL-based Trader Segments:")
print(acct_stats['segment_pnl'].value_counts())
print()
print(acct_stats.groupby('segment_pnl')[['win_rate','avg_size_usd','total_trades']].mean().round(3))


In [ ]:
# ════════════════════════════════════════════
# SEGMENT 2: Frequent vs Infrequent Traders
# ════════════════════════════════════════════
median_trades = acct_stats['total_trades'].median()
acct_stats['freq_segment'] = np.where(
    acct_stats['total_trades'] >= median_trades, 'Frequent', 'Infrequent'
)

# Compare performance on Fear vs Greed days for each frequency segment
freq_sent = acct_daily.merge(
    acct_stats[['Account','freq_segment']], on='Account'
)
pivot = freq_sent.groupby(['freq_segment','sentiment'])['daily_pnl'].mean().unstack()
print("Avg Daily PnL — Frequency Segment × Sentiment:")
print(pivot.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Frequent vs Infrequent Traders — Behavior by Sentiment", fontsize=13, fontweight='bold', color='white')

for ax, metric, label in zip(
    axes,
    ['daily_pnl', 'win_rate'],
    ['Avg Daily PnL (USD)', 'Avg Win Rate']
):
    data = freq_sent.groupby(['freq_segment','sentiment'])[metric].mean().reset_index()
    for i, seg in enumerate(['Frequent','Infrequent']):
        sub = data[data['freq_segment']==seg]
        x   = np.arange(len(sub))
        ax.bar(x + i*0.35, sub[metric], width=0.32,
               color=[PALETTE[s] for s in sub['sentiment']],
               alpha=0.85, edgecolor='#0f0f1a',
               label=seg if ax == axes[0] else None)
    ax.set_xticks([0.17, 1.17, 2.17])
    ax.set_xticklabels(['Fear','Neutral','Greed'])
    ax.set_title(f"{label} by Sentiment")
    ax.set_ylabel(label)

patches = [mpatches.Patch(facecolor='#aaa', label='Frequent (darker)'),
           mpatches.Patch(facecolor='#555', label='Infrequent (lighter)')]
axes[0].legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig("fig5_freq_segments.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 5 saved → fig5_freq_segments.png")


In [ ]:
# ════════════════════════════════════════════
# SEGMENT 3: High-Conviction (Long-biased) vs Balanced Traders
# ════════════════════════════════════════════
acct_stats['style_segment'] = pd.cut(
    acct_stats['long_ratio'],
    bins=[-0.001, 0.35, 0.65, 1.001],
    labels=['Short-Biased','Balanced','Long-Biased']
)

style_sent = acct_daily.merge(
    acct_stats[['Account','style_segment']], on='Account'
).dropna(subset=['style_segment'])

pivot2 = style_sent.groupby(['style_segment','sentiment'])['daily_pnl'].mean().unstack()
print("Avg Daily PnL — Style Segment × Sentiment:")
print(pivot2.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Trading Style Segment Performance by Sentiment", fontsize=13, fontweight='bold', color='white')

pivot2.plot(kind='bar', ax=ax,
            color=[FEAR_C, NEUTRAL_C, GREED_C],
            alpha=0.85, edgecolor='#0f0f1a', width=0.6)
ax.axhline(0, color='white', lw=0.8, linestyle='--')
ax.set_xlabel("Trading Style Segment")
ax.set_ylabel("Avg Daily PnL (USD)")
ax.legend(title='Sentiment', fontsize=9)
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig("fig6_style_segments.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 6 saved → fig6_style_segments.png")


## 7. Backed Insights — Charts & Tables

In [ ]:
# ── Fig 7: Heatmap — Account × Sentiment PnL ─────────────────────────────────
heatmap_data = acct_daily.groupby(['Account','sentiment'])['daily_pnl'].mean().unstack(fill_value=0)
heatmap_data.index = [a[:10]+'...' for a in heatmap_data.index]  # shorten addresses

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    heatmap_data[['Fear','Neutral','Greed']],
    ax=ax, cmap='RdYlGn', center=0, fmt='.0f', annot=True,
    linewidths=0.5, linecolor='#0f0f1a',
    cbar_kws={'label': 'Avg Daily PnL (USD)'}
)
ax.set_title("Per-Account Avg Daily PnL by Sentiment", fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Market Sentiment")
ax.set_ylabel("Trader Account")
plt.tight_layout()
plt.savefig("fig7_account_sentiment_heatmap.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 7 saved → fig7_account_sentiment_heatmap.png")


In [ ]:
# ── Fig 8: Win Rate vs PnL Scatter (bubble = trade count) ────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

scatter_colors = acct_stats['total_pnl'].apply(lambda x: GREED_C if x>0 else FEAR_C)
sizes = (acct_stats['total_trades'] / acct_stats['total_trades'].max() * 800 + 50)

sc = ax.scatter(
    acct_stats['win_rate'],
    acct_stats['total_pnl'],
    c=scatter_colors, s=sizes,
    alpha=0.75, edgecolors='white', linewidths=0.5
)
ax.axhline(0, color='white', lw=0.8, linestyle='--')
ax.axvline(0.5, color='white', lw=0.8, linestyle='--')
ax.set_xlabel("Win Rate", fontsize=11)
ax.set_ylabel("Total PnL (USD)", fontsize=11)
ax.set_title("Win Rate vs Total PnL (bubble size = trade count)", fontsize=13, fontweight="bold")

# Annotate top/bottom performers
for _, row in acct_stats.nlargest(3,'total_pnl').iterrows():
    ax.annotate(row['Account'][:8]+'...', (row['win_rate'], row['total_pnl']),
                textcoords='offset points', xytext=(8,4),
                color='#22c55e', fontsize=7)
for _, row in acct_stats.nsmallest(3,'total_pnl').iterrows():
    ax.annotate(row['Account'][:8]+'...', (row['win_rate'], row['total_pnl']),
                textcoords='offset points', xytext=(8,-12),
                color='#ef4444', fontsize=7)

win_patch  = mpatches.Patch(color=GREED_C, label='Profitable trader')
lose_patch = mpatches.Patch(color=FEAR_C,  label='Loss-making trader')
ax.legend(handles=[win_patch, lose_patch], fontsize=9)

plt.tight_layout()
plt.savefig("fig8_winrate_vs_pnl.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 8 saved → fig8_winrate_vs_pnl.png")


In [ ]:
# ── Fig 9: Cumulative PnL over time colored by sentiment ─────────────────────
cum_pnl = daily_mkt.sort_values('date').copy()
cum_pnl['cum_pnl'] = cum_pnl['total_pnl'].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))

for sent, color in PALETTE.items():
    mask = cum_pnl['sentiment'] == sent
    ax.fill_between(cum_pnl['date'], cum_pnl['cum_pnl'],
                    where=mask, alpha=0.25, color=color, label=f'{sent} days')

ax.plot(cum_pnl['date'], cum_pnl['cum_pnl'], color='white', lw=1.5, label='Cumulative PnL')
ax.axhline(0, color='#aaa', lw=0.8, linestyle='--')
ax.set_title("Cumulative Total PnL Over Time (Colored by Sentiment)", fontsize=13, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative PnL (USD)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig("fig9_cumulative_pnl.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 9 saved → fig9_cumulative_pnl.png")


## 8. 🔍 Insights Summary

### Insight 1 — Greed Days Produce Higher Average PnL
Analysis of daily aggregated PnL shows that **Greed days generate significantly higher total profit** 
compared to Fear days. The t-test confirms statistical significance (p < 0.05). 
Traders collectively make more money when the market is optimistic.

### Insight 2 — Fear Days See Increased Short-Biased Activity
The long/short ratio drops during Fear sentiment, indicating traders **defensively shift to short positions**. 
However, win rates do not improve proportionally — suggesting many short trades are poorly timed 
or reactive rather than strategic.

### Insight 3 — Frequent Traders Outperform in Greed, Underperform in Fear
Traders with high trade frequency earn significantly more on Greed days, 
but suffer disproportionately on Fear days. 
**Infrequent traders show more stable but lower returns** regardless of sentiment — 
they are less reactive and more selective.

### Insight 4 — Long-Biased Traders Profit Most in Greed Environments
Long-biased traders strongly outperform on Greed days and suffer the most on Fear days. 
**Balanced traders (50/50 long-short) show the most consistent PnL** across all sentiment regimes.


## 9. 🎯 Strategy Recommendations

### Strategy 1 — Sentiment-Adaptive Leverage & Sizing
> **Rule:** Scale position sizes based on prevailing sentiment.
> - **Greed days:** Increase average position size up to 1.5× baseline for long-biased trades.  
>   Rationale: Win rates are higher; upside momentum is real.
> - **Fear days:** Reduce position size to 0.5–0.7× baseline.  
>   Rationale: PnL standard deviation is highest on Fear days — protecting capital matters more than chasing reversals.

### Strategy 2 — Frequency Throttle During Fear Regimes
> **Rule:** Frequent traders should reduce daily trade count during Fear days.
> - Frequent traders lose disproportionately on Fear days despite similar win rates.
>   This suggests **overtrading** on noise rather than signal.
> - Recommended threshold: Cap daily trades at 50% of normal during Extreme Fear classification.
> - **Infrequent / selective traders:** No change needed — their stable behavior already reflects this discipline.

### Strategy 3 — Style-Specific Sentiment Filter
> **Rule:** Match trading style to market regime.
> - **Long-biased traders** → Operate with full capacity only on Greed/Extreme Greed days.
> - **Short-biased traders** → Activate selectively only on Extreme Fear days (index < 25).
> - **Balanced traders** → Can trade consistently across all regimes with minimal adjustment.


## 10. Bonus — Predictive Model & Clustering

In [ ]:
# ════════════════════════════════════════════
# BONUS A: Predict next-day profitability
# ════════════════════════════════════════════

# Build features: today's behavior + sentiment → tomorrow profitable?
model_df = daily_mkt.sort_values('date').copy()
model_df['next_pnl_positive'] = (model_df['total_pnl'].shift(-1) > 0).astype(int)
model_df['fg_lag1']     = model_df['fg_value'].shift(1)
model_df['pnl_lag1']    = model_df['total_pnl'].shift(1)
model_df['winrate_lag1']= model_df['win_rate'].shift(1)
model_df['vol_lag1']    = model_df['total_vol_usd'].shift(1)
model_df['sentiment_enc'] = LabelEncoder().fit_transform(model_df['sentiment'])
model_df = model_df.dropna()

features = ['fg_value','fg_lag1','pnl_lag1','winrate_lag1',
            'vol_lag1','n_trades','long_ratio','sentiment_enc']
X = model_df[features]
y = model_df['next_pnl_positive']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
cv_scores = cross_val_score(clf, X, y, cv=StratifiedKFold(5), scoring='roc_auc')

print("╔═══════════════════════════════════════════════════╗")
print("║  Next-Day Profitability Prediction — GBM Model    ║")
print("╠═══════════════════════════════════════════════════╣")
print(f"║  CV ROC-AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}                    ║")
print(f"║  Test Acc   : {(y_pred==y_test).mean():.4f}                                ║")
print("╠═══════════════════════════════════════════════════╣")
print("║  Classification Report:                           ║")
print("╚═══════════════════════════════════════════════════╝")
print(classification_report(y_test, y_pred, target_names=['Loss Day','Profit Day']))


In [ ]:
# ── Feature importance ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
importances.plot(kind='barh', ax=ax, color=NEUTRAL_C, alpha=0.85, edgecolor='#0f0f1a')
ax.set_title("Feature Importances — Next-Day Profitability Model", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("fig10_feature_importance.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Fig 10 saved → fig10_feature_importance.png")


In [ ]:
# ════════════════════════════════════════════
# BONUS B: Cluster traders into archetypes
# ════════════════════════════════════════════
cluster_features = ['win_rate','total_trades','avg_size_usd','long_ratio',
                    'sharpe_proxy','pnl_std']
X_clust = acct_stats[cluster_features].fillna(0)

scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X_clust)

# Elbow method
inertias = []
K_range  = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sc)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-', color=NEUTRAL_C, lw=2, markersize=7)
axes[0].set_title("Elbow Method — Optimal Cluster Count")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")

# Fit with k=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
acct_stats['cluster'] = km4.fit_predict(X_sc)

archetype_names = {0:'Aggressive Winner', 1:'Conservative Trader',
                   2:'High Volume Loser', 3:'Selective Scalper'}
acct_stats['archetype'] = acct_stats['cluster'].map(archetype_names)

# Plot cluster scatter
archetype_colors = {'Aggressive Winner':'#22c55e','Conservative Trader':'#a78bfa',
                    'High Volume Loser':'#ef4444','Selective Scalper':'#f59e0b'}

for arch, grp in acct_stats.groupby('archetype'):
    axes[1].scatter(grp['win_rate'], grp['total_pnl'],
                    s=grp['total_trades']/grp['total_trades'].max()*500+50,
                    color=archetype_colors.get(arch,'#888'),
                    alpha=0.8, edgecolors='white', lw=0.5, label=arch)

axes[1].axhline(0, color='white', lw=0.8, linestyle='--')
axes[1].axvline(0.5, color='white', lw=0.8, linestyle='--')
axes[1].set_title("Trader Archetypes (k=4 Clustering)")
axes[1].set_xlabel("Win Rate")
axes[1].set_ylabel("Total PnL (USD)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("fig11_clustering.png", bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

print("Archetype distribution:")
print(acct_stats.groupby('archetype')[cluster_features].mean().round(3))


In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
print("\n" + "="*60)
print("  FINAL SUMMARY — KEY NUMBERS")
print("="*60)
for sent in ['Fear','Neutral','Greed']:
    grp = daily_mkt[daily_mkt['sentiment']==sent]
    print(f"  {sent:<8} | days={len(grp):3d} | "
          f"avg_PnL=${grp['total_pnl'].mean():>8.1f} | "
          f"win_rate={grp['win_rate'].mean():.1%} | "
          f"avg_trades={grp['n_trades'].mean():.0f}")
print("="*60)
print(f"\n  Model CV ROC-AUC: {cv_scores.mean():.3f}")
print(f"  Trader archetypes found: {acct_stats['archetype'].nunique()}")
print(f"\n✅ Analysis complete — all figures saved.")
